# FIFA World Cup 2026 — Notebook 05: Tournament Simulation

## About

**Purpose:** Simulate the World Cup from the group stage onward to estimate each team's chance of progressing and winning.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-07<br>
**Notes:** **Phase A** — one seeded run end-to-end (72 group matches → 12 tables → Round-of-32 field), saved for inspection. A single run is NOT a forecast; its surprises are variance. **Phase B** — the knockout bracket plus a 10,000-run Monte Carlo that averages out the variance into champion %, reach-final %, and per-round odds. Knockout ties are resolved by win-probability (no draws). Tie-breakers: points → goal difference → goals for (head-to-head omitted). The match engine (scoreline grid, sampler, win probability) and the official Round-of-32 bracket + third-place assignment now live in the shared `match_engine.py`.<br>
**Description:** Reuses the Poisson + Dixon–Coles engine (notebook 03 / `match_engine.py`) and the fixtures (notebook 04). NOTE: this is the original pre-Elo strength model — superseded as the forecast by notebook 08 (Elo); kept for the narrative.

### Change Control

| Date       | Version | Author      | Changes                                              |
|------------|---------|-------------|------------------------------------------------------|
| 2026-06-07 | 0.1     | Ganapathy K | Phase A: one run group stage → RO32 field            |
| 2026-06-07 | 0.2     | Ganapathy K | Display all results + tables; save run to disk       |
| 2026-06-07 | 1.0     | Ganapathy K | Phase B: knockout bracket + 10k Monte Carlo champion% |
| 2026-06-09 | 1.1     | Ganapathy K | Shared match_engine: Dixon–Coles + official bracket  |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import sys
import pandas as pd
import numpy as np
from scipy.stats import poisson
from functools import lru_cache
from pathlib import Path

sys.path.append(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson")
from match_engine import (win_probability as engine_win_probability,
                          build_scoreline_sampler, sample_scorelines,
                          sample_scoreline, build_round_of_32)

pd.set_option("display.max_rows", 100)

### 1.2 Config

In [3]:
PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
TEAM_STRENGTHS_PATH = PROCESSED_DATA_DIR / "team_strengths.parquet"
GROUPS_PATH = PROCESSED_DATA_DIR / "wc_groups.parquet"
GROUP_FIXTURES_PATH = PROCESSED_DATA_DIR / "wc_group_fixtures.parquet"

ONE_RUN_RESULTS_PATH = PROCESSED_DATA_DIR / "one_run_group_results.parquet"
ONE_RUN_TABLES_PATH = PROCESSED_DATA_DIR / "one_run_group_tables.parquet"
CHAMPION_ODDS_PATH = PROCESSED_DATA_DIR / "champion_odds.parquet"

BASELINE_GOALS_PER_GAME = 1.370
MAX_GOALS = 10
RANDOM_SEED = 2026
N_SIMULATIONS = 10000

## 2. Load Inputs

In [4]:
team_strengths = pd.read_parquet(TEAM_STRENGTHS_PATH)
groups_table = pd.read_parquet(GROUPS_PATH)
group_fixtures = pd.read_parquet(GROUP_FIXTURES_PATH)

team_to_group = dict(zip(groups_table["team"], groups_table["group"]))
print(f"{len(team_strengths)} strengths | {len(groups_table)} teams | {len(group_fixtures)} fixtures")

336 strengths | 48 teams | 72 fixtures


## 3. Match Simulation

`expected_goals` is the notebook-03 formula (re-declared so this notebook stands alone). `simulate_match` *draws* an actual scoreline from each side's Poisson — the dice-roll that makes every run different.

In [5]:
def expected_goals(home_team, away_team):
    home = team_strengths.loc[home_team]
    away = team_strengths.loc[away_team]
    expected_home_goals = home["attack_strength"] * away["defence_strength"] * BASELINE_GOALS_PER_GAME
    expected_away_goals = away["attack_strength"] * home["defence_strength"] * BASELINE_GOALS_PER_GAME
    return expected_home_goals, expected_away_goals


def simulate_match(home_team, away_team, rng):
    expected_home_goals, expected_away_goals = expected_goals(home_team, away_team)
    return sample_scoreline(expected_home_goals, expected_away_goals, rng)

## 4. Simulate the Group Stage (one run)

Play all 72 fixtures once. The full table of this seed's scorelines is shown below.

In [6]:
rng = np.random.default_rng(RANDOM_SEED)

results = []
for fixture in group_fixtures.itertuples(index=False):
    home_goals, away_goals = simulate_match(fixture.home_team, fixture.away_team, rng)
    results.append({"group": fixture.group, "home_team": fixture.home_team,
                    "home_goals": home_goals, "away_goals": away_goals,
                    "away_team": fixture.away_team})

group_results = pd.DataFrame(results)
print(f"Simulated {len(group_results)} group matches")
group_results

Simulated 72 group matches


,group,home_team,home_goals,away_goals,away_team
0,A,Mexico,0,1,South Africa
1,A,Mexico,1,3,South Korea
2,A,Mexico,1,1,Czech Republic
3,A,South Africa,1,0,South Korea
4,A,South Africa,1,0,Czech Republic
5,A,South Korea,2,3,Czech Republic
6,B,Canada,3,3,Bosnia and Herzegovina
7,B,Canada,0,2,Qatar
8,B,Canada,2,0,Switzerland
9,B,Bosnia and Herzegovina,1,0,Qatar


## 5. Build the Group Tables

3 points a win, 1 a draw; rank each group by points → goal difference → goals for. All 12 tables below.

In [7]:
def build_group_tables(group_results):
    stats = {team: {"group": grp, "played": 0, "won": 0, "drawn": 0, "lost": 0,
                    "gf": 0, "ga": 0, "points": 0}
             for team, grp in team_to_group.items()}
    for match in group_results.itertuples(index=False):
        home, away = stats[match.home_team], stats[match.away_team]
        home["played"] += 1; away["played"] += 1
        home["gf"] += match.home_goals; home["ga"] += match.away_goals
        away["gf"] += match.away_goals; away["ga"] += match.home_goals
        if match.home_goals > match.away_goals:
            home["won"] += 1; home["points"] += 3; away["lost"] += 1
        elif match.home_goals < match.away_goals:
            away["won"] += 1; away["points"] += 3; home["lost"] += 1
        else:
            home["drawn"] += 1; away["drawn"] += 1
            home["points"] += 1; away["points"] += 1
    table = pd.DataFrame(stats).T.reset_index(names="team")
    table["gd"] = table["gf"] - table["ga"]
    table = table.sort_values(["group", "points", "gd", "gf"],
                              ascending=[True, False, False, False])
    table["position"] = table.groupby("group").cumcount() + 1
    return table.reset_index(drop=True)


group_tables = build_group_tables(group_results)
group_tables[["group", "position", "team", "played", "won", "drawn", "lost",
              "gf", "ga", "gd", "points"]]

,group,position,team,played,won,drawn,lost,gf,ga,gd,points
0,A,1,South Africa,3,3,0,0,3,0,3,9
1,A,2,Czech Republic,3,1,1,1,4,4,0,4
2,A,3,South Korea,3,1,0,2,5,5,0,3
3,A,4,Mexico,3,0,1,2,2,5,-3,1
4,B,1,Qatar,3,2,0,1,5,3,2,6
5,B,2,Bosnia and Herzegovina,3,1,1,1,7,7,0,4
6,B,3,Canada,3,1,1,1,5,5,0,4
7,B,4,Switzerland,3,1,0,2,6,8,-2,3
8,C,1,Morocco,3,2,0,1,6,4,2,6
9,C,2,Haiti,3,1,1,1,7,6,1,4


## 6. Fill the Round of 32

12 winners + 12 runners-up + 8 best third-placed = 32.

In [8]:
winners = group_tables[group_tables["position"] == 1]
runners_up = group_tables[group_tables["position"] == 2]
thirds = group_tables[group_tables["position"] == 3].sort_values(
    ["points", "gd", "gf"], ascending=False)
best_thirds = thirds.head(8)

print("Group winners:", winners["team"].tolist())
print("Runners-up:   ", runners_up["team"].tolist())
print("Best 8 thirds:", best_thirds["team"].tolist())
print("Eliminated thirds:", thirds.tail(4)["team"].tolist())
print(f"Round of 32 field size: {len(pd.concat([winners, runners_up, best_thirds]))}")

Group winners: ['South Africa', 'Qatar', 'Morocco', 'Australia', 'Ivory Coast', 'Japan', 'Egypt', 'Spain', 'France', 'Argentina', 'Uzbekistan', 'Croatia']
Runners-up:    ['Czech Republic', 'Bosnia and Herzegovina', 'Haiti', 'United States', 'Ecuador', 'Sweden', 'Iran', 'Uruguay', 'Senegal', 'Jordan', 'DR Congo', 'England']
Best 8 thirds: ['Belgium', 'Canada', 'Turkey', 'Iraq', 'Colombia', 'Scotland', 'Germany', 'South Korea']
Eliminated thirds: ['Algeria', 'Saudi Arabia', 'Ghana', 'Netherlands']
Round of 32 field size: 32


## 7. Save This Run

In [9]:
group_results.to_parquet(ONE_RUN_RESULTS_PATH, index=False)
group_tables.to_parquet(ONE_RUN_TABLES_PATH, index=False)
print(f"Saved one-run results and tables to {PROCESSED_DATA_DIR}")

Saved one-run results and tables to D:\Data Science\Visual Studio Code\fifa_wc_2026_poisson\data\processed


# Phase B — Knockout Bracket + Monte Carlo

Everything above was one run. Now we add the knockout stage and then run the *entire* tournament thousands of times to turn variance into stable probabilities.

## 8. Knockout Resolver

A knockout tie must produce a winner. We take the scoreline grid (as in notebook 03), read off P(team A wins) and P(team B wins), drop the draw mass, and flip a weighted coin. Because two fixed teams always have the same win probability, we **cache** it — across 10,000 simulations the same pairing is computed only once.

In [10]:
@lru_cache(maxsize=None)
def win_probability(team_a, team_b):
    """P(team_a beats team_b) in a knockout, draw mass removed."""
    expected_a, expected_b = expected_goals(team_a, team_b)
    return engine_win_probability(expected_a, expected_b)


def knockout_winner(team_a, team_b, rng):
    return team_a if rng.random() < win_probability(team_a, team_b) else team_b

## 9. Fast Group-Stage Qualifiers

The display version above builds a DataFrame every run — far too slow for 10,000 runs. This vectorised version draws all 72 scorelines at once with NumPy and returns the 32 qualifiers as **team names** (12 winners, 12 runners-up, 8 best thirds). Expected goals per fixture are precomputed once, since strengths never change.

In [11]:
team_list = list(team_to_group.keys())
team_index = {team: i for i, team in enumerate(team_list)}
group_of_index = [team_to_group[t] for t in team_list]

home_idx = np.array([team_index[t] for t in group_fixtures["home_team"]])
away_idx = np.array([team_index[t] for t in group_fixtures["away_team"]])
expected_home = np.array([expected_goals(h, a)[0]
                          for h, a in zip(group_fixtures["home_team"], group_fixtures["away_team"])])
expected_away = np.array([expected_goals(h, a)[1]
                          for h, a in zip(group_fixtures["home_team"], group_fixtures["away_team"])])

group_to_indices = {g: [i for i, gi in enumerate(group_of_index) if gi == g]
                    for g in sorted(set(group_of_index))}
group_stage_sampler = build_scoreline_sampler(expected_home, expected_away)


def simulate_qualifiers(rng):
    home_goals, away_goals = sample_scorelines(group_stage_sampler, rng)

    points = np.zeros(len(team_list)); gf = np.zeros(len(team_list)); ga = np.zeros(len(team_list))
    np.add.at(gf, home_idx, home_goals); np.add.at(gf, away_idx, away_goals)
    np.add.at(ga, home_idx, away_goals); np.add.at(ga, away_idx, home_goals)
    home_win = home_goals > away_goals; away_win = away_goals > home_goals; draw = home_goals == away_goals
    np.add.at(points, home_idx[home_win], 3); np.add.at(points, away_idx[away_win], 3)
    np.add.at(points, home_idx[draw], 1); np.add.at(points, away_idx[draw], 1)
    gd = gf - ga

    def rank_key(i):
        return (points[i], gd[i], gf[i])

    winners, runners, third_candidates = {}, {}, []
    for group, indices in group_to_indices.items():
        ordered = sorted(indices, key=rank_key, reverse=True)
        winners[group] = ordered[0]
        runners[group] = ordered[1]
        third_candidates.append((group, ordered[2]))
    best_thirds = sorted(third_candidates, key=lambda gi: rank_key(gi[1]), reverse=True)[:8]

    return ({g: team_list[i] for g, i in winners.items()},
            {g: team_list[i] for g, i in runners.items()},
            {g: team_list[i] for g, i in best_thirds})

## 10. One Full Tournament

Build the Round-of-32 bracket from the qualifiers (fixed template: 8 winner-vs-third, 4 winner-vs-runner, 4 runner-vs-runner), then play single-elimination down to the champion. We record, for each team, the deepest round it reached.

In [12]:
ROUND_NAMES = ["round_of_16", "quarter_final", "semi_final", "final", "champion"]


def simulate_tournament(rng):
    winners, runners, best_thirds = simulate_qualifiers(rng)
    bracket = build_round_of_32(winners, runners, best_thirds)

    reached = {}  # team -> deepest round index (0 = reached RO16, ... 4 = champion)
    current = bracket
    round_index = 0
    while len(current) >= 1:
        round_winners = [knockout_winner(a, b, rng) for a, b in current]
        for team in round_winners:
            reached[team] = round_index
        if len(round_winners) == 1:
            break
        current = [(round_winners[i], round_winners[i + 1]) for i in range(0, len(round_winners), 2)]
        round_index += 1
    return reached


demo_reached = simulate_tournament(np.random.default_rng(RANDOM_SEED))
print("Demo champion:", max(demo_reached, key=demo_reached.get))

Demo champion: Senegal


## 11. Run the Monte Carlo

Simulate the whole tournament `N_SIMULATIONS` times. For each team count how often it reached each milestone, then divide by the number of runs to get probabilities. *This* — not any single run — is the forecast.

In [13]:
rng = np.random.default_rng(RANDOM_SEED)

milestones = {name: {team: 0 for team in team_list} for name in ROUND_NAMES}
for _ in range(N_SIMULATIONS):
    reached = simulate_tournament(rng)
    for team, deepest in reached.items():
        for round_index in range(deepest + 1):
            milestones[ROUND_NAMES[round_index]][team] += 1

champion_odds = pd.DataFrame({name: pd.Series(counts) for name, counts in milestones.items()})
champion_odds = (champion_odds / N_SIMULATIONS * 100).round(1)
champion_odds = champion_odds.sort_values("champion", ascending=False)
champion_odds.columns = [c.replace("_", " ").title() + " %" for c in champion_odds.columns]
print(f"Ran {N_SIMULATIONS} tournaments")
champion_odds.head(20)

Ran 10000 tournaments


,Round Of 16 %,Quarter Final %,Semi Final %,Final %,Champion %
Morocco,55.7,38.1,25.2,16.0,9.9
Spain,53.6,34.5,22.0,13.0,7.3
England,55.9,34.2,19.7,11.9,6.9
Argentina,51.4,32.7,20.2,11.7,6.8
Japan,47.6,30.7,19.0,11.1,6.0
Portugal,53.5,32.6,18.6,10.7,6.0
Iran,49.4,28.1,15.9,8.7,4.7
Brazil,44.0,26.7,15.1,8.3,4.3
Algeria,46.4,27.7,16.0,8.2,4.2
Belgium,47.1,27.0,14.6,7.6,4.0


## 12. Save Champion Odds

In [14]:
champion_odds.to_parquet(CHAMPION_ODDS_PATH)
print(f"Saved champion odds -> {CHAMPION_ODDS_PATH.name}")
print("\nTop 10 title favourites:")
for team, row in champion_odds.head(10).iterrows():
    print(f"  {team:<22} {row['Champion %']:>5}%")

Saved champion odds -> champion_odds.parquet

Top 10 title favourites:
  Morocco                  9.9%
  Spain                    7.3%
  England                  6.9%
  Argentina                6.8%
  Japan                    6.0%
  Portugal                 6.0%
  Iran                     4.7%
  Brazil                   4.3%
  Algeria                  4.2%
  Belgium                  4.0%
